# Finding Harmonic Morphisms via Greedy Optimization

This notebook explores a direct-search approach to finding network coarse-grainings that approximate harmonic morphisms: rather than running a renormalization algorithm and then measuring harmonicity, we optimize the partition directly to maximize the harmonic degree.

We define a **harmonic modularity** objective that interpolates between standard modularity maximization (λ = 0) and pure harmonic degree maximization (λ → ∞):

$$Q_H = Q_{\text{mod}} + \lambda \cdot (-\text{H}_{\text{dev}})$$

where $Q_{\text{mod}}$ is the Newman–Girvan modularity and $\text{H}_{\text{dev}}$ is the harmonic deviation (standard deviation of boundary degrees across fibers — lower is more harmonic).

**Optimization strategy:** Greedy best-improvement over single-node reassignments. At each step, all possible moves of each node to any other cluster are evaluated; the best single move is accepted. Iteration stops when no improving move exists (local optimum).

**Three cases compared on a Barabási–Albert graph:**
- Harmonicity only: minimize $\text{H}_{\text{dev}}$ directly  
- λ = 0: standard modularity maximization  
- λ = 1: harmonic modularity (joint objective)

In [ ]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import random
import pandas as pd
import sys
sys.path.insert(1, '../')
from Harmonic_degree import *

## Objective functions

In [ ]:
def harm_std(Ag, clusters):
    """Negative harmonic deviation — maximizing this minimises H_dev."""
    _, _, _, std_h, _, _, _, _, _, _, _, _, _ = H_CF_cluster(Ag, clusters)
    return -std_h


def convert_to_communities(clusters):
    """Convert node->cluster dict to list of sets (NetworkX community format)."""
    communities = {}
    for node, group in clusters.items():
        communities.setdefault(group, set()).add(node)
    return list(communities.values())


def mod_std_harm(Ag, clusters, lambda_param):
    """Harmonic modularity: Q_mod + lambda * (-H_dev)."""
    communities = convert_to_communities(clusters)
    modularity = nx.community.modularity(Ag, communities)
    return modularity + lambda_param * harm_std(Ag, clusters)

## Greedy optimizer

In [ ]:
def balanced_random_partition(Ag, k):
    """Balanced initialisation: nodes distributed evenly across k groups."""
    nodes = list(Ag.nodes())
    random.shuffle(nodes)
    return {node: i % k for i, node in enumerate(nodes)}


def greedy_optimization(f, Ag, k, max_iterations=100, lambda_param=None):
    """
    Greedy best-improvement search over single-node reassignments.

    Parameters
    ----------
    f : callable
        Objective to maximise. Signature: f(Ag, partition) or
        f(Ag, partition, lambda_param) if lambda_param is not None.
    Ag : nx.Graph
    k : int  — number of clusters
    max_iterations : int
    lambda_param : float or None

    Returns
    -------
    dict : node -> cluster id
    """
    partition = balanced_random_partition(Ag, k)
    value = f(Ag, partition, lambda_param) if lambda_param is not None else f(Ag, partition)

    for _ in range(max_iterations):
        best_partition = partition
        best_value = value
        improved = False

        for node in Ag.nodes():
            current_group = partition[node]
            for new_group in range(k):
                if new_group == current_group:
                    continue
                candidate = {**partition, node: new_group}
                candidate_value = (
                    f(Ag, candidate, lambda_param)
                    if lambda_param is not None
                    else f(Ag, candidate)
                )
                if candidate_value > best_value:
                    best_value = candidate_value
                    best_partition = candidate
                    improved = True

        if not improved:
            break
        partition = best_partition
        value = best_value

    return partition

## Visualisation helper

In [ ]:
def plot_partition(Ag, clusters, title, layout, seed=42):
    """Show fine-grained graph (coloured by cluster) and coarse-grained graph side by side."""
    _, deg_h, M_deg_h, std_h, _, _, _, deg_cf, M_deg_cf, std_cf, _, _, _ = H_CF_cluster(Ag, clusters)
    df = pd.DataFrame(
        [[deg_h, M_deg_h, std_h, deg_cf, M_deg_cf, std_cf]],
        columns=["H_mean", "H_mod", "H_dev", "CF_mean", "CF_mod", "CF_dev"],
    )

    G_coarse, colors_d, sing_col = clust_plot(Ag, clusters)
    colors = [colors_d[n] for n in Ag.nodes()]
    lay2 = nx.spring_layout(G_coarse, iterations=70, seed=seed)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, pos, graph, node_colors, node_size, subtitle in [
        (axes[0], layout, Ag, colors, 100, "Fine-grained"),
        (axes[1], lay2, G_coarse, sing_col, 300, "Coarse-grained"),
    ]:
        drawn = nx.draw_networkx_nodes(graph, ax=ax, pos=pos,
                                       node_color=node_colors, node_size=node_size)
        drawn.set_edgecolor("white")
        nx.draw_networkx_edges(graph, ax=ax, pos=pos, width=1.8)
        ax.set_title(subtitle)
    fig.suptitle(title, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
    print(df.to_string(index=False))
    return df

## Test graph: Barabási–Albert (n=30, m=1)

A scale-free network with 30 nodes. We partition it into k=4 clusters.

In [ ]:
random.seed(0)
Ag = nx.barabasi_albert_graph(30, 1, seed=43)
K = 4
layout = nx.spring_layout(Ag, iterations=250, seed=42)

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
nx.draw(Ag, pos=layout, with_labels=False, node_size=200, ax=ax)
ax.set_title("Barabási–Albert graph (n=30, m=1)")
plt.tight_layout()
plt.show()

## Case 1: Harmonicity only (minimise H_dev)

In [ ]:
clusters_harm = greedy_optimization(harm_std, Ag, k=K, max_iterations=10000)
df_harm = plot_partition(Ag, clusters_harm, "Harmonicity only", layout)

## Case 2: Modularity only (λ = 0)

In [ ]:
clusters_mod = greedy_optimization(mod_std_harm, Ag, k=K, max_iterations=10000, lambda_param=0)
df_mod = plot_partition(Ag, clusters_mod, "Modularity only (λ = 0)", layout)

## Case 3: Harmonic modularity (λ = 1)

In [ ]:
clusters_hmod = greedy_optimization(mod_std_harm, Ag, k=K, max_iterations=10000, lambda_param=1)
df_hmod = plot_partition(Ag, clusters_hmod, "Harmonic modularity (λ = 1)", layout)

## Summary comparison

In [ ]:
summary = pd.concat(
    [df_harm, df_mod, df_hmod],
    keys=["Harmonicity only", "Modularity only (λ=0)", "Harmonic modularity (λ=1)"],
).droplevel(1)
summary.index.name = "Method"
print(summary.to_string())